# Lesson 17: Llama 3.1 8B QLoRA Fine-Tuning

Цей notebook робить повний цикл домашнього завдання:

1. Завантажує `train.jsonl` і `eval.jsonl`.
2. Проганяє baseline на Llama 3.1 8B без нашого fine-tuning.
3. Робить QLoRA fine-tuning через Unsloth.
4. Повторно проганяє eval на fine-tuned model.
5. Зберігає metrics JSON і LoRA adapter.

Перед запуском увімкни GPU: `Runtime -> Change runtime type -> T4 GPU`.

## 1. Install dependencies

Unsloth офіційно рекомендує починати з QLoRA і 4-bit для доступного fine-tuning. У цьому notebook використовується Llama 3.1 8B 4-bit checkpoint.

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q datasets trl peft accelerate bitsandbytes transformers

## 2. Upload data

Запусти клітинку і завантаж два файли з локальної папки `data`:

- `train.jsonl`
- `eval.jsonl`

In [ ]:
from pathlib import Path
from google.colab import files

Path("data").mkdir(exist_ok=True)
uploaded = files.upload()

for filename, content in uploaded.items():
    target = Path("data") / filename
    target.write_bytes(content)
    print(f"Saved {target} ({len(content)} bytes)")

## 3. Shared eval code

Тут визначаємо schema, prompt, JSON parsing і метрики. Цей самий eval використовується для baseline і fine-tuned model.

In [ ]:
import json
import statistics
import time
from collections import defaultdict
from pathlib import Path

import torch

SYSTEM_PROMPT = (
    "You extract structured data from customer support emails. "
    "Return only a single valid JSON object with fields: "
    "customer_name (string or null), product (string), "
    "issue_category (one of: billing, technical, account, feature_request, other), "
    "urgency (one of: low, medium, high, critical), "
    "summary (one short sentence). No extra text."
)

FIELDS = ["customer_name", "product", "issue_category", "urgency", "summary"]

def build_prompt(email: str) -> str:
    return f"""{SYSTEM_PROMPT}

Email:
{email}

JSON:
"""

def safe_json_parse(text: str):
    text = (text or "").strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text
        text = text.rsplit("```", 1)[0].strip()
        if text.startswith("json"):
            text = text[4:].strip()
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        text = text[start:end + 1]
    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        return None, False

def field_match(predicted, expected, field: str) -> bool:
    if field == "summary":
        if not isinstance(predicted, str) or not isinstance(expected, str):
            return False
        p_words = {w.lower().strip(".,!?:;()") for w in predicted.split() if len(w) > 3}
        e_words = {w.lower().strip(".,!?:;()") for w in expected.split() if len(w) > 3}
        if not p_words or not e_words:
            return False
        return len(p_words & e_words) / max(len(e_words), 1) >= 0.4
    if field == "customer_name":
        if predicted is None and expected is None:
            return True
        if predicted is None or expected is None:
            return False
        return expected.split()[0].lower() in str(predicted).lower()
    if predicted is None:
        return False
    return str(predicted).strip().lower() == str(expected).strip().lower()

def percentile(values, p: float):
    if not values:
        return None
    values = sorted(values)
    index = min(len(values) - 1, int(round((len(values) - 1) * p)))
    return values[index]

def load_eval(path="data/eval.jsonl"):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

@torch.inference_mode()
def generate_json(model, tokenizer, email: str, max_new_tokens: int = 220):
    prompt = build_prompt(email)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    started = time.perf_counter()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    latency = time.perf_counter() - started
    new_tokens = output_ids[0][input_len:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return raw, input_len, len(new_tokens), latency

def evaluate_model(model, tokenizer, output_path: str, model_label: str):
    examples = load_eval()
    details = []
    json_valid = 0
    exact_match = 0
    field_correct = defaultdict(int)
    input_tokens = []
    output_tokens = []
    latencies = []

    for i, example in enumerate(examples, 1):
        email = example["email"]
        expected = example["expected"]
        raw, in_tok, out_tok, latency = generate_json(model, tokenizer, email)
        predicted, valid = safe_json_parse(raw)
        if valid:
            json_valid += 1

        per_field = {}
        all_match = True
        for field in FIELDS:
            ok = valid and field_match(predicted.get(field) if predicted else None, expected[field], field)
            per_field[field] = ok
            if ok:
                field_correct[field] += 1
            else:
                all_match = False

        if valid and all_match:
            exact_match += 1

        input_tokens.append(in_tok)
        output_tokens.append(out_tok)
        latencies.append(latency)
        details.append({
            "i": i,
            "email": email,
            "expected": expected,
            "raw_output": raw,
            "predicted": predicted,
            "valid_json": valid,
            "exact_match": valid and all_match,
            "field_match": per_field,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "latency_sec": latency,
        })
        marker = "OK" if valid and all_match else "MISS"
        print(f"[{i:02d}/{len(examples)}] {marker} valid={valid} exact={valid and all_match} latency={latency:.2f}s")

    n = len(examples)
    metrics = {
        "model": model_label,
        "n_examples": n,
        "json_valid_rate": round(json_valid / n, 4),
        "exact_match_rate": round(exact_match / n, 4),
        "field_accuracy": {field: round(field_correct[field] / n, 4) for field in FIELDS},
        "avg_input_tokens": round(statistics.mean(input_tokens), 2),
        "avg_output_tokens": round(statistics.mean(output_tokens), 2),
        "latency_p50_sec": round(percentile(latencies, 0.50), 4),
        "latency_p95_sec": round(percentile(latencies, 0.95), 4),
        "details": details,
    }

    Path("results").mkdir(exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(json.dumps({k: v for k, v in metrics.items() if k != "details"}, indent=2, ensure_ascii=False))
    print(f"Saved: {output_path}")
    return metrics


## 4. Load base Llama 3.1 8B

У цьому notebook `base` означає: модель до нашого fine-tuning. Якщо треба строго foundation/base checkpoint, залиш `unsloth/llama-3.1-8b-unsloth-bnb-4bit`. Якщо Colab/HF доступ не дає завантажити модель, треба прийняти Llama license на Hugging Face або замінити model name на доступний Unsloth Llama 3.1 8B checkpoint.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/llama-3.1-8b-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Loaded", MODEL_NAME)

## 5. Baseline eval

Це метрики Llama 3.1 8B до fine-tuning. Файл треба потім скачати і покласти локально в `results/base_8b_metrics.json`.

In [ ]:
base_metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    output_path="results/base_8b_metrics.json",
    model_label=MODEL_NAME,
)

## 6. Prepare training dataset

`train.jsonl` вже лежить у chat-format. Для base checkpoint ми перетворюємо його у простий text completion format: system instruction + email + expected JSON.

In [ ]:
from datasets import Dataset

def train_row_to_text(row):
    system = row["messages"][0]["content"]
    email = row["messages"][1]["content"]
    answer = row["messages"][2]["content"]
    eos = tokenizer.eos_token or ""
    return {
        "text": f"""{system}

Email:
{email}

JSON:
{answer}{eos}"""
    }

train_rows = []
with open("data/train.jsonl", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            train_rows.append(train_row_to_text(json.loads(line)))

train_dataset = Dataset.from_list(train_rows)
print(train_dataset)
print(train_dataset[0]["text"][:800])

## 7. QLoRA fine-tuning

Параметри з домашнього завдання: 4-bit, LoRA r=16, alpha=32, 3 epochs.

In [ ]:
import shutil
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

shutil.rmtree("outputs", ignore_errors=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="no",
        report_to="none",
    ),
)

train_result = trainer.train()
print(train_result)

## 8. Save LoRA adapter

Це adapter weights, які треба зберегти як артефакт домашки.

In [ ]:
ADAPTER_DIR = "llama31_8b_email_extractor_lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter:", ADAPTER_DIR)

## 9. Fine-tuned eval

Використовуємо той самий `eval.jsonl`, що і для baseline.

In [ ]:
FastLanguageModel.for_inference(model)

finetuned_metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    output_path="results/finetuned_8b_metrics.json",
    model_label=f"{MODEL_NAME}+email-extractor-lora",
)

## 10. Download artifacts

Скачай metrics і adapter zip. Локально поклади JSON-файли в `results/`.

In [ ]:
!zip -q -r llama31_8b_email_extractor_lora.zip llama31_8b_email_extractor_lora

from google.colab import files
files.download("results/base_8b_metrics.json")
files.download("results/finetuned_8b_metrics.json")
files.download("llama31_8b_email_extractor_lora.zip")